# Clasificación de Series de Tiempo con RNN
## Reconocimiento de Actividad Humana (HAR) con Acelerómetro de Smartphone

**Curso:** Inteligencia Artificial — UAO  
**Proyecto:** Clasificación de actividades humanas usando Redes Neuronales Recurrentes  

---

### Contexto

El reconocimiento de actividad humana (Human Activity Recognition, HAR) es una aplicación clásica del aprendizaje automático en series temporales. Consiste en identificar la actividad que está realizando una persona a partir de los datos capturados por sensores inerciales (acelerómetro, giroscopio) integrados en dispositivos como smartphones o wearables.

### Dataset

Se utilizaron datos recolectados con un smartphone mediante la plataforma **Edge Impulse**, capturando señales del acelerómetro de 3 ejes (accX, accY, accZ) con un intervalo de muestreo de 16 ms (~62.5 Hz). Cada muestra corresponde a una ventana de aproximadamente 5 segundos (~312 lecturas).

**5 clases de actividad:**

| Clase | Descripción | Muestras |
|-------|-------------|----------|
| **Agitar** | Movimiento rápido y repetitivo del dispositivo | 21 |
| **Caminando** | Desplazamiento caminando con el dispositivo | 5 |
| **Girar** | Rotación del dispositivo sobre sí mismo | 21 |
| **Inclinación** | Cambio de orientación angular del dispositivo | 19 |
| **Quieto** | Dispositivo en reposo sobre superficie estable | 19 |

> ⚠️ **Nota sobre desbalance:** La clase "Caminando" tiene solo 5 muestras, lo que introduce un desbalance significativo. Esto se maneja con split estratificado y se discute en las conclusiones.

### Objetivo

Comparar el rendimiento de dos arquitecturas de redes recurrentes:
1. **SimpleRNN (Vanilla)** — RNN básica con retroalimentación simple
2. **LSTM** — RNN con memoria larga y corta plazo

Se evaluará cómo cada arquitectura captura las dependencias temporales en señales de aceleración para clasificar las 5 actividades.

## 1. Configuración e Instalación de Dependencias

La siguiente celda instala las bibliotecas necesarias. En Google Colab, descomentar la primera línea para instalar.

In [ ]:
# ============================================================
# Instalación de dependencias (descomentar en Google Colab)
# ============================================================
!pip install tensorflow matplotlib seaborn scikit-learn pydot graphviz

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU'))} dispositivo(s)")

In [ ]:
# 🔧 Ejecutar en Google Colab
!git clone https://github.com/caesar-dat-com/IA-RNN-Clasificacion.git
import os
os.chdir('IA-RNN-Clasificacion')
print("✅ Repo clonado y directorio listo")

## 2. Carga del Dataset

### Instrucciones para Google Colab

**Opción A — Clonar el repositorio:**
```python
!git clone https://github.com/caesar-dat-com/IA-RNN-Clasificacion.git
DATA_DIR = 'IA-RNN-Clasificacion/data/edge_impulse_export/training'
```

**Opción B — Subir ZIP manualmente:**
```python
from google.colab import files
uploaded = files.upload()  # Subir el archivo ZIP del dataset
!unzip -q *.zip -d dataset
DATA_DIR = 'dataset/data/edge_impulse_export/training'
```

**Opción C — Montar Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/IA-RNN-Clasificacion/data/edge_impulse_export/training'
```

Descomentar la opción que prefieras y ajustar `DATA_DIR`.

In [ ]:
# ============================================================
# Configuración del directorio de datos
# ============================================================
# Ajustar según la opción de carga elegida:
DATA_DIR = '../data/edge_impulse_export/training'

# Descomentar para Google Colab (Opción A - clonar repo):
# DATA_DIR = 'IA-RNN-Clasificacion/data/edge_impulse_export/training'

# Directorio para guardar diagramas
DOCS_DIR = '../docs'
os.makedirs(DOCS_DIR, exist_ok=True)

print(f"Directorio de datos: {DATA_DIR}")
print(f"Archivos JSON encontrados: {len([f for f in os.listdir(DATA_DIR) if f.endswith('.json')])}")

In [ ]:
# ============================================================
# Función para cargar los datos desde archivos JSON de Edge Impulse
# ============================================================

def cargar_dataset(data_dir):
    """
    Carga el dataset desde archivos JSON de Edge Impulse.
    
    Cada archivo tiene la estructura:
    {
        "payload": {
            "interval_ms": 16,
            "sensors": [{"name": "accX", "units": "m/s2"}, ...],
            "values": [[accX, accY, accZ], ...]
        }
    }
    
    El label se extrae del nombre del archivo (formato: Clase.hashid.ingestion-xxx.json)
    """
    samples = []
    labels = []
    filenames = []
    
    # Obtener todos los archivos JSON ordenados
    json_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.json')])
    
    for filename in json_files:
        filepath = os.path.join(data_dir, filename)
        
        # Extraer label del nombre del archivo (antes del primer punto)
        label = filename.split('.')[0]
        
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        # Extraer la serie temporal del payload
        values = np.array(data['payload']['values'])
        
        samples.append(values)
        labels.append(label)
        filenames.append(filename)
    
    return samples, labels, filenames


# Cargar el dataset
samples_raw, labels_raw, filenames = cargar_dataset(DATA_DIR)

# Mostrar estadísticas
print(f"Total de muestras cargadas: {len(samples_raw)}")
print(f"\nDistribución de clases:")
from collections import Counter
distribucion = Counter(labels_raw)
for clase, count in sorted(distribucion.items()):
    print(f"  {clase}: {count} muestras")

print(f"\nLongitudes de las series temporales:")
longitudes = [len(s) for s in samples_raw]
print(f"  Mínimo: {min(longitudes)}")
print(f"  Máximo: {max(longitudes)}")
print(f"  Media: {np.mean(longitudes):.1f}")
print(f"  Mediana: {np.median(longitudes):.1f}")

print(f"\nDimensiones por muestra: {samples_raw[0].shape[1]} (accX, accY, accZ)")
print(f"Intervalo de muestreo: 16 ms (~62.5 Hz)")

In [ ]:
# ============================================================
# Visualización de ejemplos por clase
# ============================================================

nombres_ejes = ['accX', 'accY', 'accZ']
colores_ejes = ['#e74c3c', '#2ecc71', '#3498db']

clases_unicas = sorted(set(labels_raw))
fig, axes = plt.subplots(len(clases_unicas), 1, figsize=(14, 3*len(clases_unicas)), sharex=True)

for i, clase in enumerate(clases_unicas):
    # Tomar el primer ejemplo de cada clase
    idx = labels_raw.index(clase)
    tiempo_ms = np.arange(len(samples_raw[idx])) * 16  # intervalo de 16ms
    
    for j in range(3):
        axes[i].plot(tiempo_ms, samples_raw[idx][:, j], 
                     label=nombres_ejes[j], color=colores_ejes[j], alpha=0.8)
    
    axes[i].set_ylabel(f'{clase}\n(m/s²)', fontsize=10)
    axes[i].legend(loc='upper right', fontsize=8)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Tiempo (ms)', fontsize=12)
fig.suptitle('Ejemplos de señales de acelerómetro por clase', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'ejemplos_senales.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocesamiento

El preprocesamiento incluye los siguientes pasos:

1. **Padding/Truncado:** Las series temporales tienen longitudes variables. Se truncan o se rellenan con ceros a una longitud fija (`MAX_LEN = 312`, la longitud más común).
2. **Normalización:** Se estandarizan las características usando la media y desviación estándar del conjunto de entrenamiento (z-score normalization).
3. **Codificación de etiquetas:** Se convierten las etiquetas de texto a enteros y luego a one-hot encoding.
4. **Split estratificado:** Se divide el dataset en 80% entrenamiento y 20% prueba, manteniendo la proporción de clases.

In [ ]:
# ============================================================
# Preprocesamiento: Padding/Truncado + Normalización
# ============================================================

# Longitud fija para todas las series (la mayoría tienen 312)
MAX_LEN = 312
N_FEATURES = 3  # accX, accY, accZ

def pad_truncate(samples, max_len):
    """
    Trunca o rellena con ceros cada serie temporal a max_len.
    - Si la serie es más larga: se trunca al inicio (se conservan los últimos max_len valores)
    - Si la serie es más corta: se rellena con ceros al final
    """
    result = []
    for s in samples:
        if len(s) >= max_len:
            # Truncar: conservar los últimos max_len pasos
            result.append(s[-max_len:])
        else:
            # Padding con ceros al final
            pad = np.zeros((max_len - len(s), s.shape[1]))
            result.append(np.vstack([s, pad]))
    return np.array(result)


# Aplicar padding/truncado
X_padded = pad_truncate(samples_raw, MAX_LEN)
print(f"Shape después de padding/truncado: {X_padded.shape}")
print(f"  Muestras: {X_padded.shape[0]}")
print(f"  Pasos temporales: {X_padded.shape[1]}")
print(f"  Características: {X_padded.shape[2]}")

In [ ]:
# ============================================================
# Codificación de etiquetas
# ============================================================

# Mapeo de clases a enteros
clases = sorted(set(labels_raw))
label_to_idx = {label: idx for idx, label in enumerate(clases)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print("Mapeo de etiquetas:")
for label, idx in label_to_idx.items():
    print(f"  {label} → {idx}")

# Convertir etiquetas a enteros
y_int = np.array([label_to_idx[l] for l in labels_raw])

# One-hot encoding
N_CLASSES = len(clases)
y_onehot = to_categorical(y_int, num_classes=N_CLASSES)

print(f"\nShape de etiquetas one-hot: {y_onehot.shape}")
print(f"Número de clases: {N_CLASSES}")

In [ ]:
# ============================================================
# Split estratificado 80/20
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot,
    test_size=0.2,
    random_state=SEED,
    stratify=y_int  # Mantener proporción de clases
)

# También necesitamos las etiquetas enteras para métricas
y_train_int = np.argmax(y_train, axis=1)
y_test_int = np.argmax(y_test, axis=1)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")
print(f"\nDistribución en entrenamiento:")
for idx in range(N_CLASSES):
    count = np.sum(y_train_int == idx)
    print(f"  {idx_to_label[idx]}: {count}")
print(f"\nDistribución en prueba:")
for idx in range(N_CLASSES):
    count = np.sum(y_test_int == idx)
    print(f"  {idx_to_label[idx]}: {count}")

In [ ]:
# ============================================================
# Normalización z-score (basada en entrenamiento)
# ============================================================

# Calcular media y desviación estándar del conjunto de entrenamiento
# Shape: (n_train, max_len, 3) → stats por feature (eje)
mean = X_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, 3)
std = X_train.std(axis=(0, 1), keepdims=True)    # (1, 1, 3)

# Aplicar normalización a ambos conjuntos
X_train_norm = (X_train - mean) / (std + 1e-8)  # epsilon para evitar división por cero
X_test_norm = (X_test - mean) / (std + 1e-8)

print("Normalización z-score aplicada:")
for i, nombre in enumerate(nombres_ejes):
    print(f"  {nombre}: media={mean.flatten()[i]:.4f}, std={std.flatten()[i]:.4f}")

print(f"\nShape final de entrenamiento: {X_train_norm.shape}")
print(f"Shape final de prueba: {X_test_norm.shape}")
print(f"Media del set normalizado: {X_train_norm.mean():.6f}")
print(f"Std del set normalizado: {X_train_norm.std():.6f}")

## 4. Modelo RNN Simple (Vanilla)

La **SimpleRNN** es la forma más básica de red neuronal recurrente. En cada paso temporal $t$, la unidad calcula:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

Donde:
- $x_t$: entrada en el paso $t$ (vector de 3 features)
- $h_{t-1}$: estado oculto del paso anterior
- $W_{xh}$: pesos de entrada a oculto
- $W_{hh}$: pesos de oculto a oculto (recurrentes)
- $b_h$: sesgo

**Parámetros de una capa SimpleRNN:**

$$\text{params} = \text{units} \times (\text{units} + \text{input\_dim} + 1)$$

La limitación principal de SimpleRNN es el problema del **gradiente desvaneciente**, que dificulta aprender dependencias a largo plazo.

In [ ]:
# ============================================================
# Modelo 1: SimpleRNN (Vanilla)
# ============================================================

# Hiperparámetros
RNN_UNITS = 64
DROPOUT_RATE = 0.3
LEARNING_RATE = 0.001
BATCH_SIZE = 16
EPOCHS = 100
PATIENCE = 10

# Construir el modelo SimpleRNN
model_rnn = Sequential([
    SimpleRNN(RNN_UNITS, input_shape=(MAX_LEN, N_FEATURES), 
              name='simple_rnn_1'),
    Dropout(DROPOUT_RATE, name='dropout_1'),
    Dense(32, activation='relu', name='dense_1'),
    Dropout(DROPOUT_RATE, name='dropout_2'),
    Dense(N_CLASSES, activation='softmax', name='output')
], name='SimpleRNN_Classifier')

# Compilar
model_rnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Mostrar resumen
model_rnn.summary()

In [ ]:
# ============================================================
# Entrenamiento del modelo SimpleRNN
# ============================================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

history_rnn = model_rnn.fit(
    X_train_norm, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ============================================================
# Gráficas de entrenamiento — SimpleRNN
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history_rnn.history['accuracy'], label='Entrenamiento', linewidth=2)
ax1.plot(history_rnn.history['val_accuracy'], label='Validación', linewidth=2)
ax1.set_title('SimpleRNN — Accuracy por Epoch', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history_rnn.history['loss'], label='Entrenamiento', linewidth=2)
ax2.plot(history_rnn.history['val_loss'], label='Validación', linewidth=2)
ax2.set_title('SimpleRNN — Loss por Epoch', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'rnn_simple_training.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMejor val_accuracy: {max(history_rnn.history['val_accuracy']):.4f}")
print(f"Mejor val_loss: {min(history_rnn.history['val_loss']):.4f}")
print(f"Épocas entrenadas: {len(history_rnn.history['loss'])}")

## 5. Modelo LSTM

La **LSTM (Long Short-Term Memory)** fue diseñada para resolver el problema del gradiente desvaneciente. Incorpora tres puertas (gates) que controlan el flujo de información:

1. **Puerta de olvido (forget gate):** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ — Decide qué información descartar del estado de celda.
2. **Puerta de entrada (input gate):** $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ — Decide qué nueva información almacenar.
3. **Puerta de salida (output gate):** $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ — Decide qué parte del estado producir como salida.

**Parámetros de una capa LSTM:**

$$\text{params} = 4 \times \text{units} \times (\text{units} + \text{input\_dim} + 1)$$

El factor 4 se debe a las 4 transformaciones lineales (forget gate, input gate, candidate cell, output gate), cada una con sus propios pesos y sesgos.

In [ ]:
# ============================================================
# Modelo 2: LSTM
# ============================================================

# Usamos los mismos hiperparámetros para comparación justa
LSTM_UNITS = RNN_UNITS  # Misma cantidad de unidades para comparar

# Construir el modelo LSTM
model_lstm = Sequential([
    LSTM(LSTM_UNITS, input_shape=(MAX_LEN, N_FEATURES), 
         name='lstm_1'),
    Dropout(DROPOUT_RATE, name='dropout_1'),
    Dense(32, activation='relu', name='dense_1'),
    Dropout(DROPOUT_RATE, name='dropout_2'),
    Dense(N_CLASSES, activation='softmax', name='output')
], name='LSTM_Classifier')

# Compilar
model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Mostrar resumen
model_lstm.summary()

In [ ]:
# ============================================================
# Entrenamiento del modelo LSTM
# ============================================================

early_stop_lstm = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

history_lstm = model_lstm.fit(
    X_train_norm, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop_lstm],
    verbose=1
)

In [ ]:
# ============================================================
# Gráficas de entrenamiento — LSTM
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history_lstm.history['accuracy'], label='Entrenamiento', linewidth=2)
ax1.plot(history_lstm.history['val_accuracy'], label='Validación', linewidth=2)
ax1.set_title('LSTM — Accuracy por Epoch', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history_lstm.history['loss'], label='Entrenamiento', linewidth=2)
ax2.plot(history_lstm.history['val_loss'], label='Validación', linewidth=2)
ax2.set_title('LSTM — Loss por Epoch', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'lstm_training.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMejor val_accuracy: {max(history_lstm.history['val_accuracy']):.4f}")
print(f"Mejor val_loss: {min(history_lstm.history['val_loss']):.4f}")
print(f"Épocas entrenadas: {len(history_lstm.history['loss'])}")

## 6. Validación y Comparación

Evaluamos ambos modelos en el conjunto de prueba usando:
- **Classification report** (precision, recall, f1-score por clase)
- **Matriz de confusión**
- **Tabla comparativa** de métricas globales

In [ ]:
# ============================================================
# Evaluación en el conjunto de prueba
# ============================================================

# Predicciones
y_pred_rnn = model_rnn.predict(X_test_norm)
y_pred_lstm = model_lstm.predict(X_test_norm)

# Convertir a etiquetas enteras
y_pred_rnn_int = np.argmax(y_pred_rnn, axis=1)
y_pred_lstm_int = np.argmax(y_pred_lstm, axis=1)

# Nombres de clases para los reportes
target_names = [idx_to_label[i] for i in range(N_CLASSES)]

print("="*60)
print("REPORTE DE CLASIFICACIÓN — SimpleRNN")
print("="*60)
print(classification_report(y_test_int, y_pred_rnn_int, 
                          target_names=target_names, zero_division=0))

print("\n" + "="*60)
print("REPORTE DE CLASIFICACIÓN — LSTM")
print("="*60)
print(classification_report(y_test_int, y_pred_lstm_int, 
                          target_names=target_names, zero_division=0))

In [ ]:
# ============================================================
# Matrices de confusión
# ============================================================

cm_rnn = confusion_matrix(y_test_int, y_pred_rnn_int)
cm_lstm = confusion_matrix(y_test_int, y_pred_lstm_int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# SimpleRNN
sns.heatmap(cm_rnn, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax1)
ax1.set_title('SimpleRNN — Matriz de Confusión', fontsize=13)
ax1.set_ylabel('Etiqueta Real')
ax1.set_xlabel('Predicción')

# LSTM
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=target_names, yticklabels=target_names, ax=ax2)
ax2.set_title('LSTM — Matriz de Confusión', fontsize=13)
ax2.set_ylabel('Etiqueta Real')
ax2.set_xlabel('Predicción')

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Tabla comparativa de métricas
# ============================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calcular métricas para cada modelo
metrics_rnn = {
    'Accuracy': accuracy_score(y_test_int, y_pred_rnn_int),
    'Precision (macro)': precision_score(y_test_int, y_pred_rnn_int, average='macro', zero_division=0),
    'Recall (macro)': recall_score(y_test_int, y_pred_rnn_int, average='macro', zero_division=0),
    'F1-Score (macro)': f1_score(y_test_int, y_pred_rnn_int, average='macro', zero_division=0),
}

metrics_lstm = {
    'Accuracy': accuracy_score(y_test_int, y_pred_lstm_int),
    'Precision (macro)': precision_score(y_test_int, y_pred_lstm_int, average='macro', zero_division=0),
    'Recall (macro)': recall_score(y_test_int, y_pred_lstm_int, average='macro', zero_division=0),
    'F1-Score (macro)': f1_score(y_test_int, y_pred_lstm_int, average='macro', zero_division=0),
}

# También métricas weighted
metrics_rnn['Precision (weighted)'] = precision_score(y_test_int, y_pred_rnn_int, average='weighted', zero_division=0)
metrics_rnn['Recall (weighted)'] = recall_score(y_test_int, y_pred_rnn_int, average='weighted', zero_division=0)
metrics_rnn['F1-Score (weighted)'] = f1_score(y_test_int, y_pred_rnn_int, average='weighted', zero_division=0)

metrics_lstm['Precision (weighted)'] = precision_score(y_test_int, y_pred_lstm_int, average='weighted', zero_division=0)
metrics_lstm['Recall (weighted)'] = recall_score(y_test_int, y_pred_lstm_int, average='weighted', zero_division=0)
metrics_lstm['F1-Score (weighted)'] = f1_score(y_test_int, y_pred_lstm_int, average='weighted', zero_division=0)

# Parámetros del modelo
metrics_rnn['Parámetros'] = model_rnn.count_params()
metrics_lstm['Parámetros'] = model_lstm.count_params()

# Épocas entrenadas
metrics_rnn['Épocas'] = len(history_rnn.history['loss'])
metrics_lstm['Épocas'] = len(history_lstm.history['loss'])

# Mostrar tabla
print("\n" + "="*70)
print("TABLA COMPARATIVA — SimpleRNN vs LSTM")
print("="*70)
print(f"{'Métrica':<25} {'SimpleRNN':>12} {'LSTM':>12}")
print("-" * 50)
for key in metrics_rnn:
    if key in ['Parámetros', 'Épocas']:
        print(f"{key:<25} {metrics_rnn[key]:>12} {metrics_lstm[key]:>12}")
    else:
        print(f"{key:<25} {metrics_rnn[key]:>12.4f} {metrics_lstm[key]:>12.4f}")
print("="*70)

# Determinar el mejor modelo
mejor_rnn = metrics_rnn['F1-Score (macro)']
mejor_lstm = metrics_lstm['F1-Score (macro)']
ganador = 'LSTM' if mejor_lstm > mejor_rnn else 'SimpleRNN' if mejor_rnn > mejor_lstm else 'Empate'
print(f"\nMejor modelo por F1-Score (macro): {ganador}")
print(f"  SimpleRNN: {mejor_rnn:.4f}")
print(f"  LSTM: {mejor_lstm:.4f}")

In [ ]:
# ============================================================
# Gráfica comparativa lado a lado
# ============================================================

# Métricas por clase
from sklearn.metrics import precision_score, recall_score, f1_score

precision_rnn = precision_score(y_test_int, y_pred_rnn_int, average=None, zero_division=0)
precision_lstm = precision_score(y_test_int, y_pred_lstm_int, average=None, zero_division=0)
recall_rnn = recall_score(y_test_int, y_pred_rnn_int, average=None, zero_division=0)
recall_lstm = recall_score(y_test_int, y_pred_lstm_int, average=None, zero_division=0)
f1_rnn = f1_score(y_test_int, y_pred_rnn_int, average=None, zero_division=0)
f1_lstm = f1_score(y_test_int, y_pred_lstm_int, average=None, zero_division=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
x = np.arange(N_CLASSES)
width = 0.35

for ax, metric_rnn, metric_lstm, title in [
    (axes[0], precision_rnn, precision_lstm, 'Precision'),
    (axes[1], recall_rnn, recall_lstm, 'Recall'),
    (axes[2], f1_rnn, f1_lstm, 'F1-Score')
]:
    bars1 = ax.bar(x - width/2, metric_rnn, width, label='SimpleRNN', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x + width/2, metric_lstm, width, label='LSTM', color='#2ecc71', alpha=0.8)
    ax.set_xlabel('Clase')
    ax.set_ylabel(title)
    ax.set_title(f'{title} por Clase', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(target_names, rotation=30, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'comparacion_metricas.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Diagramas de Arquitectura

Visualizamos la arquitectura de ambos modelos usando `tf.keras.utils.plot_model`.

In [ ]:
# ============================================================
# Diagramas de arquitectura
# ============================================================

try:
    # Diagrama SimpleRNN
    rnn_arch_path = os.path.join(DOCS_DIR, 'rnn_simple_arch.png')
    tf.keras.utils.plot_model(
        model_rnn,
        to_file=rnn_arch_path,
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        dpi=150
    )
    print(f"Diagrama SimpleRNN guardado en: {rnn_arch_path}")
    
    # Diagrama LSTM
    lstm_arch_path = os.path.join(DOCS_DIR, 'lstm_arch.png')
    tf.keras.utils.plot_model(
        model_lstm,
        to_file=lstm_arch_path,
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        dpi=150
    )
    print(f"Diagrama LSTM guardado en: {lstm_arch_path}")
    
except Exception as e:
    print(f"No se pudieron generar los diagramas: {e}")
    print("Nota: Se requiere pydot y graphviz instalados.")
    print("En Colab: !apt-get install graphviz && !pip install pydot")

In [ ]:
# ============================================================
# Mostrar diagramas en el notebook
# ============================================================

from IPython.display import Image, display

rnn_arch_path = os.path.join(DOCS_DIR, 'rnn_simple_arch.png')
lstm_arch_path = os.path.join(DOCS_DIR, 'lstm_arch.png')

if os.path.exists(rnn_arch_path):
    print("\nArquitectura — SimpleRNN:")
    display(Image(filename=rnn_arch_path))

if os.path.exists(lstm_arch_path):
    print("\nArquitectura — LSTM:")
    display(Image(filename=lstm_arch_path))

## 8. Cálculo Detallado de Parámetros por Capa

### Fórmulas de parámetros

Para una capa recurrente con `units` neuronas y `input_dim` dimensiones de entrada:

**SimpleRNN:**
$$\text{params} = \text{units} \times (\text{units} + \text{input\_dim} + 1)$$

Desglose:
- $W_{xh}$: pesos de entrada → `units × input_dim`
- $W_{hh}$: pesos recurrentes → `units × units`
- $b_h$: sesgos → `units`
- Total: `units × input_dim + units × units + units = units × (units + input_dim + 1)`

**LSTM:**
$$\text{params} = 4 \times \text{units} \times (\text{units} + \text{input\_dim} + 1)$$

El factor 4 viene de las 4 puertas/camino internos: forget gate ($f_t$), input gate ($i_t$), candidate cell ($\tilde{c}_t$), y output gate ($o_t$). Cada una tiene sus propios pesos $W$ y sesgos $b$.

**Dense:**
$$\text{params} = \text{input\_dim} \times \text{units} + \text{units}$$

(pesos + sesgos)

In [ ]:
# ============================================================
# Cálculo detallado de parámetros por capa
# ============================================================

def calcular_params_simplernn(units, input_dim):
    """Parámetros de una capa SimpleRNN."""
    return units * (units + input_dim + 1)

def calcular_params_lstm(units, input_dim):
    """Parámetros de una capa LSTM."""
    return 4 * units * (units + input_dim + 1)

def calcular_params_dense(input_dim, units):
    """Parámetros de una capa Dense."""
    return input_dim * units + units

# Extraer información de las capas
print("="*75)
print("MODELO SimpleRNN — Cálculo detallado de parámetros")
print("="*75)
print(f"{'Capa':<20} {'Tipo':<12} {'Unidades':<10} {'Input Dim':<12} {'Fórmula':<35} {'Params':>8}")
print("-"*75)

total_rnn = 0
for layer in model_rnn.layers:
    layer_type = type(layer).__name__
    layer_units = layer.units if hasattr(layer, 'units') else 0
    
    if layer_type == 'SimpleRNN':
        input_dim = layer.input_shape[-1]
        params = calcular_params_simplernn(layer_units, input_dim)
        formula = f"{layer_units} × ({layer_units} + {input_dim} + 1)"
    elif layer_type == 'Dense':
        input_dim = layer.input_shape[-1]
        params = calcular_params_dense(input_dim, layer_units)
        formula = f"{input_dim} × {layer_units} + {layer_units}"
    elif layer_type == 'Dropout':
        params = 0
        formula = "—"
    else:
        params = layer.count_params()
        formula = "—"
    
    total_rnn += params
    print(f"{layer.name:<20} {layer_type:<12} {layer_units:<10} {getattr(layer, 'input_shape', ['?'])[-1] if hasattr(layer, 'input_shape') else '?':<12} {formula:<35} {params:>8}")

print("-"*75)
print(f"{'TOTAL':<20} {'':<12} {'':<10} {'':<12} {'':<35} {total_rnn:>8}")
print(f"\nVerificación Keras: {model_rnn.count_params()} parámetros")

print("\n")

print("="*75)
print("MODELO LSTM — Cálculo detallado de parámetros")
print("="*75)
print(f"{'Capa':<20} {'Tipo':<12} {'Unidades':<10} {'Input Dim':<12} {'Fórmula':<35} {'Params':>8}")
print("-"*75)

total_lstm = 0
for layer in model_lstm.layers:
    layer_type = type(layer).__name__
    layer_units = layer.units if hasattr(layer, 'units') else 0
    
    if layer_type == 'LSTM':
        input_dim = layer.input_shape[-1]
        params = calcular_params_lstm(layer_units, input_dim)
        formula = f"4 × {layer_units} × ({layer_units} + {input_dim} + 1)"
    elif layer_type == 'Dense':
        input_dim = layer.input_shape[-1]
        params = calcular_params_dense(input_dim, layer_units)
        formula = f"{input_dim} × {layer_units} + {layer_units}"
    elif layer_type == 'Dropout':
        params = 0
        formula = "—"
    else:
        params = layer.count_params()
        formula = "—"
    
    total_lstm += params
    print(f"{layer.name:<20} {layer_type:<12} {layer_units:<10} {getattr(layer, 'input_shape', ['?'])[-1] if hasattr(layer, 'input_shape') else '?':<12} {formula:<35} {params:>8}")

print("-"*75)
print(f"{'TOTAL':<20} {'':<12} {'':<10} {'':<12} {'':<35} {total_lstm:>8}")
print(f"\nVerificación Keras: {model_lstm.count_params()} parámetros")

In [ ]:
# ============================================================
# Tabla comparativa de parámetros
# ============================================================

print("\n" + "="*60)
print("COMPARACIÓN DE PARÁMETROS")
print("="*60)
print(f"{'Concepto':<40} {'SimpleRNN':>10} {'LSTM':>10}")
print("-"*60)

# Capas recurrentes
rnn_params_rec = calcular_params_simplernn(RNN_UNITS, N_FEATURES)
lstm_params_rec = calcular_params_lstm(LSTM_UNITS, N_FEATURES)
print(f"{'Capa recurrente (units=' + str(RNN_UNITS) + ', input=' + str(N_FEATURES) + ')':<40} {rnn_params_rec:>10} {lstm_params_rec:>10}")
print(f"{'Factor multiplicativo':<40} {'×1':>10} {'×4':>10}")
print(f"{'Razón LSTM/SimpleRNN (capa recurrente)':<40} {1:>10} {lstm_params_rec/rnn_params_rec:>10.1f}")
print(f"\n{'Total de parámetros del modelo':<40} {model_rnn.count_params():>10} {model_lstm.count_params():>10}")
print(f"{'Razón total LSTM/SimpleRNN':<40} {1:>10} {model_lstm.count_params()/model_rnn.count_params():>10.2f}")
print("="*60)

print("\nAnálisis:")
print(f"  - La capa LSTM tiene ~4× más parámetros que la SimpleRNN con las mismas unidades.")
print(f"  - Esto se debe a las 4 puertas internas (forget, input, candidate, output).")
print(f"  - SimpleRNN ({RNN_UNITS} unidades): {rnn_params_rec:,} parámetros recurrentes")
print(f"  - LSTM ({LSTM_UNITS} unidades): {lstm_params_rec:,} parámetros recurrentes")
print(f"  - Ratio: {lstm_params_rec/rnn_params_rec:.1f}x")

## 9. Conclusiones

### Resultados clave

1. **SimpleRNN vs LSTM:** Se compararon ambas arquitecturas con el mismo número de unidades (64) y estructura de capas densas.

2. **Parámetros:** La LSTM tiene aproximadamente 4× más parámetros en su capa recurrente que la SimpleRNN equivalente, debido a sus 4 puertas internas.

3. **Rendimiento:** Los resultados varían según la complejidad de las dependencias temporales en los datos.

### Desafíos del dataset

- **Desbalance de clases:** La clase "Caminando" tiene solo 5 muestras, lo que dificulta su aprendizaje y evaluación. Un split estratificado garantiza al menos 1 muestra en test, pero esto es insuficiente para una evaluación confiable.
- **Dataset pequeño:** Con solo 85 muestras totales, el modelo tiene datos limitados para generalizar.
- **Soluciones posibles:** Data augmentation para series temporales, recolección de más datos, o uso de pesos de clase (`class_weight`).

### Recomendaciones

- **Aumentar datos de Caminando:** Recolectar al menos 15-20 muestras adicionales.
- **Data augmentation:** Técnicas como jittering, scaling, y window slicing para series temporales.
- **Modelos más complejos:** Probar BiLSTM, GRU, o arquitecturas híbridas (CNN+LSTM).
- **Validación cruzada:** Con tan pocos datos, K-fold cross-validation es más robusta que un simple split.

In [ ]:
# ============================================================
# Guardar modelos entrenados
# ============================================================

model_rnn.save(os.path.join('../models', 'rnn_simple_model.h5'))
model_lstm.save(os.path.join('../models', 'lstm_model.h5'))

print("Modelos guardados:")
print(f"  SimpleRNN: ../models/rnn_simple_model.h5")
print(f"  LSTM: ../models/lstm_model.h5")